In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from  pyspark.sql.classic.dataframe import DataFrame
from pyspark.sql.window import Window

In [12]:
spark = (SparkSession.builder
         .appName("ML project with spark")
         .getOrCreate()
         )
df = spark.read.csv("../../data_sample/ecommerce_clickstream_transactions.csv", header=True, inferSchema=True)


In [15]:



def transform_column_name(dataframe:DataFrame)->DataFrame:
    column_name_transformation={
        'UserId':'user_id',
        'SessionId':'session_id',
        'Timestamp':'timestamp',
        'EventType':'event_type',
        'ProductId':'product_id',
        'Amount':'amount',
        'Outcome':'purchased',
    }
    for key , value in column_name_transformation.items():
        dataframe=dataframe.withColumnRenamed(key,value)
    return dataframe


In [16]:
def eliminate_invalid_event_type(dataframe:DataFrame)->DataFrame:

    valid_events=[
        'logout',
        'page_view',
        'login',
        'purchase',
        'product_view',
        'add_to_cart',
        'click']
    dataframe=dataframe.withColumn(
        'event_type',
        when(
            col('event_type').isin(valid_events),
            col('event_type')
        )
        .otherwise('unknown')
    )

    return dataframe

In [17]:
def user_and_session_filtering(dataframe:DataFrame)->DataFrame:
    dataframe= dataframe.filter(
        (col('user_id').isNotNull()) & (col('session_id').isNotNull())
    )
    return dataframe

In [19]:
def clean_and_sort_timestamp(dataframe:DataFrame)->DataFrame:

    dataframe =dataframe.filter(
        (col('timestamp').isNotNull())
    )

    dataframe = dataframe.orderBy(
        'user_id',
        'session_id',
        'timestamp',
    )

    return dataframe

In [20]:
def boolean_map_to_purchased(dataframe:DataFrame)->DataFrame:

    dataframe =dataframe.withColumn(
        'purchased',
        when(col('purchased')=='purchase',1)
        .otherwise(0)
        .cast('int')
    )
    return dataframe

In [25]:
def purchased_user_amount_filter(dataframe:DataFrame)->DataFrame:

    dataframe =dataframe.filter(
        (col('amount')<0) |
        (col('purchased')==1 & col('amount')==0) |
        (col('purchased')==0 & col('amount')>0)
    )
    return dataframe


In [ ]:



def filter_impossible_user_journey(dataframe:DataFrame)->DataFrame:

    invalid_events_after_logout=[
        'page_view',
        'click',
        'product_view',
        'add_to_cart',
        'purchase',
    ]

    window =(
        Window
             .partitionBy('user_id','session_id')
             .orderBy('timestamp')
    )

    dataframe = dataframe.withColumn('previous_event',
                                     lag('event_type').over(window))
    dataframe=dataframe.withColumn(
        "invalid_journey",
        when(
            (col('previous_event')=='logout') &
            (col('event_type').isin(invalid_events_after_logout)) ,
            1
        ).otherwise(0)
    )

    invalid_sessions =(
        dataframe.filter(col("invalid_journey")==1)
        .select("user_id","session_id")
        .distinct()
    )

    dataframe =(
        dataframe
        .join(
            invalid_sessions,
            ['user_id','session_id'],
            'left_anti'
        )
    )

    dataframe=dataframe.drop('previous_event', 'invalid_journey')
    return dataframe

In [ ]:
def remove_duplicate_events(dataframe:DataFrame)->DataFrame:
    matchers=[
        'user_id',
        'session_id',
        'timestamp',
        'event_type',
        'product_id'
    ]
    return dataframe.drop_duplicates(matchers)

In [26]:
def aggregate_user_features(dataframe:DataFrame)->DataFrame:

    user_features=(
        dataframe.groupBy("user_id")
        .agg(
            count(when(col('event_type')=='page_view',True))
            .alise("total_view"),

            count(when(col('event_type')=='click',True))
            .alise("total_click"),

            count(when(col('event_type')=='add_to_cart',True))
            .alise("total_cart_adds"),

            count_distinct("session_id").alias("total_sessions"),
            count_distinct("product_id").alias("unique_products")

        )
    )

    session_time=(
        dataframe
        .groupby('user_id','session_id')
        .agg(
            (
                unix_timestamp(max('timestamp')) -
                unix_timestamp(min('timestamp'))
            ).alias('session_duration')
        ).groupBy('user_id')
        .agg(
            avg('session_duration').alias('session_duration_time'),
        )
    )

    user_features=(
        user_features
        .join(session_time,'user_id','left')
    )

    return user_features



In [21]:
df=transform_column_name(df)
df=eliminate_invalid_event_type(df)
df=user_and_session_filtering(df)
df=clean_and_sort_timestamp(df)
df=boolean_map_to_purchased(df)
df=purchased_user_amount_filter(df)
df=filter_impossible_user_journey(df)
df=aggregate_user_features(df)

In [23]:

df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- purchased: integer (nullable = false)

